# Degrau de Potencial sob Controle Difusional Reversível

## Bibliotecas Necessárias:

In [ ]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from scipy import special
from ipywidgets import Text, Button, HBox, VBox, Output
from IPython.display import display, clear_output

## Definindo o Modelo:

In [ ]:
model = pybamm.BaseModel()

co = pybamm.Variable("Concentração de O", domain="electrolyte")
cr = pybamm.Variable("Concentração de R", domain="electrolyte")

eta = pybamm.Parameter("Sobrepotencial Aplicado [V]")
F   = pybamm.Parameter("Constante de Faraday [C.mol-1]")
R   = pybamm.Parameter("Constante dos Gases [J.K-1.mol-1]")
T   = pybamm.Parameter("Temperatura [K]")

No = -pybamm.grad(co)   # fluxo difusional de O
Nr = -pybamm.grad(cr)   # fluxo difusional de R

model.rhs = {
    co: -pybamm.div(No),  # lei de Fick para O (eq. 1.0a)
    cr: -pybamm.div(Nr),  # lei de Fick para R (eq. 1.0b)
}

# condições iniciais (eqs. 1.1a e 1.1b)
model.initial_conditions = {
    co: pybamm.Scalar(1),  # c_O(x,0) = 1: O uniforme e adimensionalizado
    cr: pybamm.Scalar(0),  # c_R(x,0) = 0: R inicialmente ausente
}

# condições de contorno — Dirichlet
# esquerda (x=0): interface eletrodo/solução — equilíbrio de Nernst
# direita  (x=6): seio da solução            — difusão semi-infinita
f    = F / (R * T)         # fator de Nernst: f = F/(RT)
teta = pybamm.exp(f * eta) # parâmetro de Nernst: θ = exp(f·η)

model.boundary_conditions = {
    co: {
        "left":  (teta / (1 + teta), "Dirichlet"),  # eq. 1.3a: c_O(0,t) = θ/(1+θ)
        "right": (pybamm.Scalar(1),  "Dirichlet"),  # eq. 1.2a: c_O(∞,t) = 1
    },
    cr: {
        "left":  (1 / (1 + teta),   "Dirichlet"),  # eq. 1.3b: c_R(0,t) = 1/(1+θ)
        "right": (pybamm.Scalar(0),  "Dirichlet"),  # eq. 1.2b: c_R(∞,t) = 0
    },
}

model.variables = {
    "Concentração de O": co,
    "Concentração de R": cr,
    "Fluxo de O": No,
    "Fluxo de R": Nr,
}

param = pybamm.ParameterValues(
    {
        "Sobrepotencial Aplicado [V]": "[input]",
        "Constante de Faraday [C.mol-1]": 96485.3,
        "Constante dos Gases [J.K-1.mol-1]": 8.31446,
        "Temperatura [K]": 298.15,
    }
)

# ── geometria e malha ────────────────────────────────────────────────────────
x_var = pybamm.SpatialVariable("x", domain=["electrolyte"], coord_sys="cartesian")

geometry = {"electrolyte": {x_var: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(6)}}}

submesh_types = {"electrolyte": pybamm.Uniform1DSubMesh}
var_pts       = {x_var: 400}
mesh          = pybamm.Mesh(geometry, submesh_types, var_pts)

spatial_methods = {"electrolyte": pybamm.FiniteVolume()}
disc = pybamm.Discretisation(mesh, spatial_methods)

param.process_model(model)
param.process_geometry(geometry)
disc.process_model(model)

## Funções Analíticas:

In [ ]:
F_NUM = 38.9217  # f = F/(RT) a 298.15 K [V⁻¹]

def teta_num(eta):
    """Parâmetro de Nernst numérico: θ = exp(f·η)"""
    return np.exp(F_NUM * eta)

def co_analitica(x, t, eta):
    """Solução analítica de c_O: c_O(x,t) = 1 - [1/(1+θ)] · erfc(x / 2√t)"""
    th = teta_num(eta)
    return 1 - (1 / (1 + th)) * special.erfc(x / (2 * np.sqrt(t)))

def cr_analitica(x, t, eta):
    """Solução analítica de c_R: c_R(x,t) = [1/(1+θ)] · erfc(x / 2√t)"""
    th = teta_num(eta)
    return (1 / (1 + th)) * special.erfc(x / (2 * np.sqrt(t)))

def i_analitica(t, eta):
    """Corrente analítica adimensional: i(t) = −1 / [√(πt)·(1+θ)]
    Negativa por convenção catódica (redução).
    """
    th = teta_num(eta)
    return -1 / (np.sqrt(np.pi * t) * (1 + th))

## Gráfico Estático:

In [ ]:
ETA_PADRAO = -0.1  # sobrepotencial padrão [V]

solver  = pybamm.ScipySolver()
t       = np.linspace(1e-5, 1, 1000)
solucao = solver.solve(model, t, inputs={"Sobrepotencial Aplicado [V]": ETA_PADRAO})

co_sol = solucao["Concentração de O"]
cr_sol = solucao["Concentração de R"]
No_sol = solucao["Fluxo de O"]

x_plot = np.linspace(0, 6, 200)
tempos = [1, 0.1, 0.01, 0.001]
cores  = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for ti, cor in zip(tempos, cores):
    # ── espécie O ──────────────────────────────────────────────────────────
    ax2.plot(
        x_plot, co_sol(t=ti, x=x_plot),
        ".", color=cor, markersize=8,
        label=f"O numérico   t = {ti}",
    )
    ax2.plot(
        x_plot, co_analitica(x_plot, ti, ETA_PADRAO),
        "-", color=cor, linewidth=2,
        label=f"O analítico  t = {ti}",
    )
    # ── espécie R ──────────────────────────────────────────────────────────
    ax2.plot(
        x_plot, cr_sol(t=ti, x=x_plot),
        "s", color=cor, markersize=5,
        label=f"R numérico   t = {ti}",
    )
    ax2.plot(
        x_plot, cr_analitica(x_plot, ti, ETA_PADRAO),
        ":", color=cor, linewidth=2,
        label=f"R analítico  t = {ti}",
    )

ax2.set_xlabel("x")
ax2.set_ylabel("Concentração")
ax2.set_xlim([0, 0.9])
ax2.set_ylim([0, 1.05])
ax2.legend(fontsize=7, ncol=2)
ax2.set_title(f"Perfis de concentração (η = {ETA_PADRAO} V)")

ax1.plot(solucao.t, No_sol(solucao.t, x=0), "r.", label="Numérico")
ax1.plot(solucao.t, i_analitica(solucao.t, ETA_PADRAO), "b-", label="Analítico")
ax1.set_xlabel("t")
ax1.set_ylabel("Corrente")
ax1.set_xlim([0.01, 1])
ax1.set_ylim([-6, 0])
ax1.legend()
ax1.set_title("Corrente vs Tempo")

plt.tight_layout()
plt.show()

## Gráfico Interativo:

In [ ]:
x_plot = np.linspace(0, 6, 200)
t_int  = np.linspace(1e-5, 1, 1000)

solver_int = pybamm.ScipySolver()

saida = Output()

def plotar(eta_str):
    with saida:
        clear_output(wait=True)

        try:
            eta_ap = float(eta_str)
        except ValueError:
            print("Por favor, insira um número válido.")
            return

        if eta_ap >= 0:
            print("Por favor, insira um sobrepotencial negativo (η < 0).")
            return

        solucao = solver_int.solve(
            model, t_int,
            inputs={"Sobrepotencial Aplicado [V]": eta_ap}
        )

        co_sol = solucao["Concentração de O"]
        cr_sol = solucao["Concentração de R"]
        No_sol = solucao["Fluxo de O"]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

        tempos = [1, 0.1, 0.01, 0.001]
        cores  = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

        for ti, cor in zip(tempos, cores):
            # ── espécie O ──────────────────────────────────────────────────
            ax2.plot(
                x_plot, co_sol(t=ti, x=x_plot),
                ".", color=cor, markersize=8,
                label=f"O numérico   t = {ti}",
            )
            ax2.plot(
                x_plot, co_analitica(x_plot, ti, eta_ap),
                "-", color=cor, linewidth=2,
                label=f"O analítico  t = {ti}",
            )
            # ── espécie R ──────────────────────────────────────────────────
            ax2.plot(
                x_plot, cr_sol(t=ti, x=x_plot),
                "s", color=cor, markersize=5,
                label=f"R numérico   t = {ti}",
            )
            ax2.plot(
                x_plot, cr_analitica(x_plot, ti, eta_ap),
                ":", color=cor, linewidth=2,
                label=f"R analítico  t = {ti}",
            )

        ax2.set_xlabel("x")
        ax2.set_ylabel("Concentração")
        ax2.set_xlim([0, 0.9])
        ax2.set_ylim([0, 1.05])
        ax2.legend(fontsize=7, ncol=2)
        ax2.set_title(f"Perfis de concentração (η = {eta_ap} V)")

        ax1.plot(solucao.t, No_sol(solucao.t, x=0), "r.", label="Numérico")
        ax1.plot(solucao.t, i_analitica(solucao.t, eta_ap), "b-", label="Analítico")
        ax1.set_xlabel("t")
        ax1.set_ylabel("Corrente")
        ax1.set_xlim([0.01, 1])
        ax1.set_ylim([-6, 0])
        ax1.legend()
        ax1.set_title("Corrente vs Tempo")

        plt.tight_layout()
        display(fig)
        plt.close(fig)

# ── widgets ──────────────────────────────────────────────────────────────────
campo_eta = Text(
    value="-0.1",
    description="Sobrepotencial η [V]  (η < 0):",
    style={"description_width": "initial"},
)

botao = Button(description="Recalcular", button_style="success")

def ao_clicar(b):
    plotar(campo_eta.value)

botao.on_click(ao_clicar)

interface = VBox([HBox([campo_eta, botao]), saida])
display(interface)

# gráfico inicial
plotar(campo_eta.value)